# Polaris RE — Cross-Jurisdiction Capital: LICAT vs US RBC vs EU Solvency II

**Purpose:** Demonstrate the cross-jurisdiction regulatory-capital epic. A
reinsurer pricing the *same* block on a return-on-capital basis must be able to
quote it under whichever standard the cedant files under — Canadian **LICAT**,
US **NAIC RBC**, or EU **Solvency II SCR**. All three plug into one
`CapitalModel` protocol, so a single `capital_model_for(...)` selector swaps the
standard with no change to the pricing path.

This notebook prices one term block under each standard and compares, side by
side:

- the **required-capital run-off** (`capital_by_period`),
- the **return on capital** (RoC) and capital-adjusted IRR, and
- the new **regulatory solvency ratio** (`available_capital / denominator`,
  ADR-103/104) — the headline number a committee reads (LICAT total ratio,
  RBC ratio, EU solvency ratio).

> **Calibration caveat (read first).** The per-standard component factors are
> *committee-stage placeholders* (documented in ADR-098 / ADR-100): LICAT's
> default C-2 is a **10% mortality shock** on NAR, whereas RBC's C-2 (0.15% of
> NAR) and Solvency II's mortality sub-factor (0.20% of NAR) are small ongoing
> factors. (The Solvency II *aggregated* SCR runs a little above its mortality
> sub-factor — ~0.28% of NAR at issue here — because it also folds in the +0.15%
> catastrophe shock via the standard-formula correlation matrices; it is the
> aggregated SCR, not the bare 0.20%, that drives the required capital below.)
> The standards are therefore **not yet mutually calibrated** — the *levels*
> differ by roughly 50–100x and are not directly comparable across standards.
> What this notebook demonstrates is
> that (a) each standard runs end-to-end through the shared selector, and (b)
> the solvency ratio behaves correctly *within* a standard (linear in available
> capital). Mutual factor calibration is the Asset/ALM epic.

In [1]:
from datetime import date
from pathlib import Path

import numpy as np

from polaris_re.analytics.capital_base import (
    SUPPORTED_CAPITAL_MODELS,
    capital_model_for,
    capital_model_label,
)
from polaris_re.analytics.profit_test import ProfitTester
from polaris_re.assumptions.assumption_set import AssumptionSet
from polaris_re.assumptions.lapse import LapseAssumption
from polaris_re.assumptions.mortality import MortalityTable, MortalityTableSource
from polaris_re.core.inforce import InforceBlock
from polaris_re.core.policy import Policy, ProductType, Sex, SmokerStatus
from polaris_re.core.projection import ProjectionConfig
from polaris_re.products.dispatch import get_product_engine
from polaris_re.utils.table_io import MortalityTableArray

## 1. Build a term inforce block

A small, homogeneous term block (all male non-smoker, new issue) keeps the
comparison easy to read. The block carries little reserve, so capital is driven
almost entirely by the Net Amount at Risk — exactly the C-2 / mortality-risk
component each standard models, which makes the cross-standard contrast clean.

In [2]:
VALUATION_DATE = date(2025, 1, 1)
AVAILABLE_CAPITAL = 900_000.0  # company-held available capital / TAC / own funds

policies = [
    Policy(
        policy_id=f"TL{i:03d}",
        issue_age=45,
        attained_age=45,
        sex=Sex.MALE,
        smoker_status=SmokerStatus.NON_SMOKER,
        underwriting_class="STANDARD",
        face_amount=500_000.0,
        annual_premium=4_200.0,
        policy_term=20,
        duration_inforce=0,
        issue_date=VALUATION_DATE,
        valuation_date=VALUATION_DATE,
        product_type=ProductType.TERM,
    )
    for i in range(10)
]
block = InforceBlock(policies=policies)
print(f"{block.n_policies} policies, total face ${block.total_face_amount():,.0f}")

10 policies, total face $5,000,000


## 2. Assumptions (flat mortality + lapse)

A flat best-estimate table keeps the example self-contained (no CSV load). The
capital factors, not the assumption table, drive the cross-standard contrast.

In [3]:
N_AGES = 121 - 18
qx = np.full(N_AGES, 0.003, dtype=np.float64).reshape(-1, 1)

tables = {}
for sex in (Sex.MALE, Sex.FEMALE):
    for smoker in (SmokerStatus.SMOKER, SmokerStatus.NON_SMOKER, SmokerStatus.UNKNOWN):
        tables[f"{sex.value}_{smoker.value}"] = MortalityTableArray(
            rates=qx.copy(), min_age=18, max_age=120, select_period=0,
            source_file=Path("synthetic"),
        )

mortality = MortalityTable(
    source=MortalityTableSource.CSO_2001, table_name="Synthetic flat",
    min_age=18, max_age=120, select_period_years=0,
    has_smoker_distinct_rates=False, tables=tables,
)
lapse = LapseAssumption.from_duration_table(
    {1: 0.06, 2: 0.06, 3: 0.06, "ultimate": 0.06}
)
assumptions = AssumptionSet(
    mortality=mortality, lapse=lapse, version="capital-standards-nb",
    effective_date=VALUATION_DATE,
)

## 3. Project gross cash flows and the Net Amount at Risk

The capital models hold capital against the **Net Amount at Risk** (NAR =
in-force face − reserve). We derive it transparently from the projection — the
same `max(face·inforce_ratio − reserve, 0)` the YRT treaty uses — and feed the
*identical* NAR to all three standards via `nar=`, so the comparison isolates
the capital factors rather than any difference in exposure.

In [4]:
config = ProjectionConfig(
    valuation_date=VALUATION_DATE,
    projection_horizon_years=20,
    discount_rate=0.06,
    maintenance_cost_per_policy_per_year=50.0,
)
engine = get_product_engine(inforce=block, assumptions=assumptions, config=config)
gross = engine.project()

# Net Amount at Risk: in-force face (approximated by premium run-off) minus reserve.
initial_premium = gross.gross_premiums[0]
inforce_ratio = gross.gross_premiums / initial_premium
nar_vec = np.maximum(
    block.total_face_amount() * inforce_ratio - gross.reserve_balance, 0.0
)
print(f"NAR at issue ${nar_vec[0]:,.0f}, peak ${nar_vec.max():,.0f}")

NAR at issue $5,000,000, peak $5,000,000


## 4. Price under each capital standard

`capital_model_for(model_id, product_type)` resolves the registry id
(`licat` / `rbc` / `solvency2`) to the calculator; `run_with_capital` joins the
profit test with the chosen standard's required-capital schedule and computes
the solvency ratio from the supplied `available_capital`. The pricing path is
identical across standards — only the model id changes.

In [5]:
results = {}
for model_id in SUPPORTED_CAPITAL_MODELS:
    model = capital_model_for(model_id, ProductType.TERM)
    results[model_id] = ProfitTester(cashflows=gross, hurdle_rate=0.10).run_with_capital(
        model, nar=nar_vec, available_capital=AVAILABLE_CAPITAL
    )

print("Priced under:", ", ".join(capital_model_label(m) for m in results))

Priced under: LICAT (Canada), US RBC, EU Solvency II


## 5. Required-capital run-off

Each standard produces a per-period required-capital schedule that grades down
with the NAR. The *shape* is comparable; the *level* reflects each standard's
placeholder factor (see the calibration caveat — LICAT's shock-scale C-2 sits
~100x above the RBC / Solvency II ongoing factors).

In [6]:
labels = ["Issue", "Year 1", "Year 5", "Year 10", "Year 15"]
months = [0, 12, 60, 120, 180]
header = f"{'Standard':<26}" + "".join(f"{lbl:>14}" for lbl in labels)
print(header)
print("-" * len(header))
for model_id, res in results.items():
    cap = res.capital_by_period
    row = "".join(f"{cap[m]:>14,.0f}" for m in months)
    print(f"{capital_model_label(model_id):<26}{row}")

Standard                           Issue        Year 1        Year 5       Year 10       Year 15
------------------------------------------------------------------------------------------------
LICAT (Canada)                   750,000       702,885       542,221       392,005       283,404
US RBC                             7,500         7,029         5,422         3,920         2,834
EU Solvency II                    13,919        13,045        10,063         7,275         5,260


## 6. Return on capital and the solvency ratio

The headline pricing metrics, side by side: peak required capital, PV of the
capital stock, return on capital (PV profits ÷ PV capital), the capital-adjusted
IRR, and the regulatory solvency ratio (`available_capital ÷ denominator`).

In [7]:
def fmt_pct(x):
    return f"{x:.1%}" if x is not None else "N/A"


header = (
    f"{'Standard':<26}{'Peak capital':>16}{'PV capital':>16}"
    f"{'RoC':>9}{'Solvency ratio':>16}"
)
print(header)
print("-" * len(header))
for model_id, res in results.items():
    print(
        f"{capital_model_label(model_id):<26}"
        f"{res.peak_capital:>16,.0f}{res.pv_capital:>16,.0f}"
        f"{fmt_pct(res.return_on_capital):>9}{fmt_pct(res.capital_ratio):>16}"
    )

print(
    "\nThe three solvency ratios apply the SAME $900,000 of available capital to "
    "each\nstandard's own denominator. Because the standards are not yet mutually "
    "calibrated\n(see the caveat), the ratio LEVELS are not comparable across "
    "standards — only\nwithin one. The next cell shows the within-standard "
    "behaviour the surface guarantees."
)

Standard                      Peak capital      PV capital      RoC  Solvency ratio
-----------------------------------------------------------------------------------
LICAT (Canada)                     750,000      53,833,087     0.3%          120.0%
US RBC                               7,500         538,331    29.4%        24000.0%
EU Solvency II                      13,919         999,100    15.9%         6465.8%

The three solvency ratios apply the SAME $900,000 of available capital to each
standard's own denominator. Because the standards are not yet mutually calibrated
(see the caveat), the ratio LEVELS are not comparable across standards — only
within one. The next cell shows the within-standard behaviour the surface guarantees.


## 7. The solvency ratio is linear in available capital (within a standard)

The property the ratio surface guarantees (and the unit tests pin): for a fixed
block and standard, the ratio scales linearly with available capital — double
the held capital, double the ratio. This is the meaningful, well-defined use of
the ratio.

In [8]:
licat = capital_model_for("licat", ProductType.TERM)
base = ProfitTester(cashflows=gross, hurdle_rate=0.10).run_with_capital(
    licat, nar=nar_vec, available_capital=AVAILABLE_CAPITAL
)
doubled = ProfitTester(cashflows=gross, hurdle_rate=0.10).run_with_capital(
    licat, nar=nar_vec, available_capital=2 * AVAILABLE_CAPITAL
)
print(f"available ${AVAILABLE_CAPITAL:>12,.0f}  ->  LICAT ratio {base.capital_ratio:.1%}")
print(f"available ${2 * AVAILABLE_CAPITAL:>12,.0f}  ->  LICAT ratio {doubled.capital_ratio:.1%}")
print(f"ratio of ratios: {doubled.capital_ratio / base.capital_ratio:.2f}x (expected 2.00x)")

available $     900,000  ->  LICAT ratio 120.0%
available $   1,800,000  ->  LICAT ratio 240.0%
ratio of ratios: 2.00x (expected 2.00x)


## 8. Takeaway

A reinsurer can now price the **same** block under any of three regulatory
capital standards by passing one selector — on the CLI
(`polaris price --capital {licat,rbc,solvency2} --available-capital N`), the API
(`capital_model` + `available_capital` on the price request), the dashboard
(capital-basis selectbox + available-capital input), or directly via
`capital_model_for(...)` + `ProfitTester.run_with_capital(...)`. Each standard
emits its required-capital schedule, return on capital, and regulatory solvency
ratio through the same jurisdiction-agnostic surface.

**What remains:** the component factors are committee-stage placeholders, so the
cross-standard required-capital *levels* are not yet mutually calibrated — that
shock-based calibration is the Asset/ALM epic. Until then, treat the
cross-standard comparison as a demonstration of the *machinery*, and the
within-standard ratio (linear in available capital) as the actuarially
meaningful output.